# 🚀 Async Programming Patterns for Agentic AI Systems

A hands-on, self-contained companion notebook: every cell is independently runnable and
uses a fake LLM + fake tools with realistic latencies, so you can measure real wall-clock
behavior without spending API credits.

## Learning Objectives
In this notebook, you will learn:
1. **`asyncio` primitives** - coroutines, `Task`, `gather`, and why sync code silently serializes an "async" agent
2. **Concurrent tool fan-out** - `asyncio.gather` vs. `asyncio.TaskGroup` (3.11+) for calling multiple tools at once
3. **Streaming** - token-by-token generation with `async for`, including pausing a stream mid-way to run a tool
4. **Concurrency control** - `Semaphore` caps, `asyncio.timeout`, retry with backoff (`tenacity`), and cancellation semantics
5. **Production patterns** - a multi-agent supervisor, fire-and-forget background logging, wrapping blocking SDKs with `asyncio.to_thread`, and a load-testing harness

## Prerequisites
- **Python 3.11+** - required for `asyncio.TaskGroup`, `asyncio.timeout`, and `except*` (`ExceptionGroup`) syntax
- Familiarity with `async`/`await` basics (see `01_Async_and_Streaming.ipynb` for the LangGraph-specific version)
- Optional: `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` / Databricks credentials in `.env` to run the real-provider section (Part 10) - every cell there gracefully skips if the corresponding credential is missing

> **The one rule that matters for the whole notebook:** anything that would block the event
> loop (`time.sleep`, `requests.get`, a sync DB driver) is either replaced with an `asyncio`
> equivalent or wrapped with `asyncio.to_thread`. Break that rule and your "async" agent
> silently runs single-threaded.

---
## 🧪 Part 0: Environment Check

You need Python 3.11+ for `asyncio.TaskGroup` and `asyncio.timeout`. If you're on 3.10 or
older, the fan-out and structured-concurrency cells in Parts 4 and 6 will fail - upgrade
the kernel first.

In [1]:
# ============================================================================
# ENVIRONMENT CHECK: Verify Python 3.11+
# ============================================================================
import sys
import platform

assert sys.version_info >= (3, 11), f"Need Python 3.11+, got {sys.version}"
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

Python: 3.12.14
Platform: Windows-11-10.0.26200-SP0


In [2]:
# ============================================================================
# OPTIONAL: Install Required Packages
# ============================================================================
# Uncomment and run if these aren't already installed - skip any you don't need
# !pip install --quiet httpx tenacity nest_asyncio openai anthropic

In [3]:
# ============================================================================
# SETUP: Nested Event Loop Patch (Jupyter-Only) and Core Imports
# ============================================================================
# Jupyter already runs an event loop, so we patch it to allow nested asyncio.run().
# In a plain .py script this patch is NOT needed.
import nest_asyncio
nest_asyncio.apply()

import asyncio
import contextlib
import logging
import random
import time
from dataclasses import dataclass
from typing import AsyncIterator, Callable

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s.%(msecs)03d | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("async-agent")
print("✅ Environment ready!")

✅ Environment ready!


---
## ⚙️ Part 1: `asyncio` Primitives - The 90-Second Refresher

### Key Concepts:
- **`async def`** defines a **coroutine function**. Calling it returns a **coroutine object** - it does NOT run yet
- **`await`** *suspends* the current coroutine and hands control back to the event loop
- **`asyncio.create_task(coro)`** schedules a coroutine to run *concurrently* and returns a `Task`
- **`asyncio.gather(*coros)`** runs many coroutines concurrently and waits for all of them
- **Blocking calls** (`time.sleep`, `requests.get`) freeze the *entire* event loop - this is the #1 async-agent bug

In [4]:
# ============================================================================
# ASYNCIO PRIMITIVES: Sequential vs. Concurrent Awaits
# ============================================================================
async def hello(name: str, delay: float) -> str:
    await asyncio.sleep(delay)  # non-blocking - yields to the event loop
    return f"hello {name} (slept {delay}s)"


async def demo_primitives():
    # --- Sequential: the awaits are serialized ---
    t0 = time.perf_counter()
    a = await hello("A", 1.0)
    b = await hello("B", 1.0)
    print(f"sequential: {time.perf_counter() - t0:.2f}s  ->  {a} | {b}")

    # --- Concurrent: both waits overlap ---
    t0 = time.perf_counter()
    a, b = await asyncio.gather(hello("A", 1.0), hello("B", 1.0))
    print(f"gather:     {time.perf_counter() - t0:.2f}s  ->  {a} | {b}")


asyncio.run(demo_primitives())

sequential: 2.02s  ->  hello A (slept 1.0s) | hello B (slept 1.0s)
gather:     1.01s  ->  hello A (slept 1.0s) | hello B (slept 1.0s)


---
## 🤖 Part 2: Fake LLM and Fake Tools

We simulate an agent workload with realistic latencies so we can measure wall time without
paying for tokens:
- `fake_llm(prompt)` - 2-4s "think" time
- `fake_llm_stream(prompt)` - yields tokens with jitter
- `web_search(q)` / `vector_search(q)` / `sql_query(q)` - 0.3-1.5s each

In [5]:
# ============================================================================
# FAKE LLM AND TOOLS: Simulated Latencies for Wall-Time Experiments
# ============================================================================
async def fake_llm(prompt: str, jitter: tuple[float, float] = (2.0, 4.0)) -> str:
    await asyncio.sleep(random.uniform(*jitter))
    return f"LLM_ANSWER(prompt={prompt[:32]!r})"


async def fake_llm_stream(prompt: str, n_tokens: int = 20) -> AsyncIterator[str]:
    for i in range(n_tokens):
        await asyncio.sleep(random.uniform(0.05, 0.15))
        yield f"tok{i} "


async def web_search(q: str) -> str:
    await asyncio.sleep(random.uniform(0.4, 1.2))
    return f"web[{q}]"


async def vector_search(q: str) -> str:
    await asyncio.sleep(random.uniform(0.3, 0.9))
    return f"vec[{q}]"


async def sql_query(q: str) -> str:
    await asyncio.sleep(random.uniform(0.6, 1.5))
    return f"sql[{q}]"


TOOLS: dict[str, Callable] = {"web": web_search, "vector": vector_search, "sql": sql_query}
print("✅ Fake LLM and tools ready:", list(TOOLS.keys()))

✅ Fake LLM and tools ready: ['web', 'vector', 'sql']


---
## ⏱️ Part 3: Sync vs. Async - The Wall-Time Trap

A 5-step agent, each step ~3s. **Sync-style = ~15s total.** **Async fan-out = ~max single step**,
because the three independent tool calls run concurrently instead of one after another.

In [23]:
# ============================================================================
# SYNC-STYLE VS. TRULY-ASYNC AGENT: Wall-Time Comparison
# ============================================================================
async def sync_style_agent():
    """Awaits sequentially - the 'sync-in-async' anti-pattern."""
    t0 = time.perf_counter()
    r1 = await fake_llm("plan")
    r2 = await web_search("databricks async")
    r3 = await vector_search("agent framework")
    r4 = await sql_query("SELECT * FROM users")
    r5 = await fake_llm("summarise")
    return time.perf_counter() - t0, [r1, r2, r3, r4, r5]


async def truly_async_agent():
    """LLM plan first, then fan out the 3 independent tools, then a final LLM call."""
    t0 = time.perf_counter()
    r1 = await fake_llm("plan")
    r2, r3, r4 = await asyncio.gather(
        web_search("databricks async"),
        vector_search("agent framework"),
        sql_query("SELECT * FROM users"),
    )
    r5 = await fake_llm("summarise")
    return time.perf_counter() - t0, [r1, r2, r3, r4, r5]


async def bench():
    s, _ = await sync_style_agent()
    print(f"sync-style   : {s:5.2f}s")
    a, _ = await truly_async_agent()
    print(f"async fan-out: {a:5.2f}s  (speed-up ~{s / a:.1f}x)")


asyncio.run(bench())

sync-style   :  9.67s
async fan-out:  7.05s  (speed-up ~1.4x)


---
## 🔀 Part 4: Parallel Tool Fan-Out - `gather` and `TaskGroup`

`asyncio.gather` is the classic. `asyncio.TaskGroup` (3.11+) is **strictly better** for agent
orchestration because:
- Exceptions propagate naturally - one failed tool cancels the whole group
- No orphan tasks left running after an error
- Structured lifetime tied to the `async with` block

In [7]:
# ============================================================================
# TOOL FAN-OUT: gather vs. TaskGroup
# ============================================================================
async def fanout_with_gather(query: str):
    results = await asyncio.gather(
        *(tool(query) for tool in TOOLS.values()), return_exceptions=True
    )
    return dict(zip(TOOLS.keys(), results))


async def fanout_with_taskgroup(query: str):
    out = {}
    async with asyncio.TaskGroup() as tg:
        tasks = {name: tg.create_task(tool(query)) for name, tool in TOOLS.items()}
    # After the block, ALL tasks are done (or the group already raised)
    return {name: t.result() for name, t in tasks.items()}


async def demo_fanout():
    t0 = time.perf_counter()
    print(await fanout_with_gather("vector db"), f"({time.perf_counter() - t0:.2f}s)")

    t0 = time.perf_counter()
    print(await fanout_with_taskgroup("vector db"), f"({time.perf_counter() - t0:.2f}s)")


asyncio.run(demo_fanout())

{'web': 'web[vector db]', 'vector': 'vec[vector db]', 'sql': 'sql[vector db]'} (1.45s)
{'web': 'web[vector db]', 'vector': 'vec[vector db]', 'sql': 'sql[vector db]'} (1.15s)


### 4.1 ⚠️ What Happens When a Tool Blows Up?

`TaskGroup` cancels its siblings automatically - no dangling coroutines left behind.

In [8]:
# ============================================================================
# TASKGROUP ERROR HANDLING: Sibling Cancellation on Failure
# ============================================================================
async def flaky_tool():
    await asyncio.sleep(0.3)
    raise RuntimeError("vector index timeout")


async def slow_tool():
    try:
        await asyncio.sleep(3.0)
        return "done"
    except asyncio.CancelledError:
        print("  slow_tool: got cancelled - cleaning up")
        raise


async def demo_taskgroup_error():
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(flaky_tool())
            tg.create_task(slow_tool())
    except* RuntimeError as eg:
        # Python 3.11 ExceptionGroup syntax
        for e in eg.exceptions:
            print("caught:", e)


asyncio.run(demo_taskgroup_error())

  slow_tool: got cancelled - cleaning up
caught: vector index timeout


---
## 🌊 Part 5: Streaming Token Generation

Streaming means sending bytes to the user **before** the agent finishes thinking. Async
iterators (`async for`) are the mechanism that makes this possible.

In [9]:
# ============================================================================
# STREAMING: Token-by-Token Output with async for
# ============================================================================
async def stream_agent(prompt: str):
    print("assistant: ", end="", flush=True)
    async for tok in fake_llm_stream(prompt):
        print(tok, end="", flush=True)
    print()


asyncio.run(stream_agent("explain async agents"))

assistant: tok0 tok1 tok2 tok3 tok4 tok5 tok6 tok7 tok8 tok9 tok10 tok11 tok12 tok13 tok14 tok15 tok16 tok17 tok18 tok19 


### 5.1 🛑 Streaming + Mid-Stream Tool Call

A more realistic agent: stream tokens, detect a `TOOL:` marker, pause the stream, run the
tool, then resume with the result folded in.

In [10]:
# ============================================================================
# STREAMING: Pausing Mid-Stream to Execute a Tool
# ============================================================================
async def scripted_stream():
    parts = [
        "Let me check that. ",
        "TOOL:web(databricks agent framework) ",
        "Based on the result, ",
        "here is the answer.",
    ]
    for p in parts:
        await asyncio.sleep(0.4)
        yield p


async def stream_with_tools():
    async for chunk in scripted_stream():
        if chunk.startswith("TOOL:"):
            call = chunk[5:].strip()
            tool_name, arg = call.split("(", 1)
            arg = arg.rstrip(") ")
            print(f"\n[executing tool {tool_name}({arg!r})]", flush=True)
            result = await TOOLS[tool_name](arg)
            print(f"[tool result: {result}]", flush=True)
        else:
            print(chunk, end="", flush=True)
    print()


asyncio.run(stream_with_tools())

Let me check that. 
[executing tool web('databricks agent framework')]
[tool result: web[databricks agent framework]]
Based on the result, here is the answer.


---
## 🚦 Part 6: Concurrency Control - Semaphore, Timeout, Retry, Cancellation

Unbounded `gather` will get you rate-limited within minutes. Cap it, time it out, retry it,
and make sure cancellation cleans up properly.

In [11]:
# ============================================================================
# CONCURRENCY CONTROL: Capping Parallelism with Semaphore
# ============================================================================
async def rate_limited_llm(prompt: str, sem: asyncio.Semaphore) -> str:
    async with sem:  # at most N concurrent
        return await fake_llm(prompt, jitter=(0.5, 1.0))


async def demo_semaphore():
    sem = asyncio.Semaphore(3)  # cap at 3 concurrent LLM calls
    t0 = time.perf_counter()
    await asyncio.gather(*(rate_limited_llm(f"q{i}", sem) for i in range(12)))
    print(f"12 calls, cap=3, elapsed={time.perf_counter() - t0:.2f}s  (~4 waves)")


asyncio.run(demo_semaphore())

12 calls, cap=3, elapsed=3.10s  (~4 waves)


In [12]:
# ============================================================================
# CONCURRENCY CONTROL: Timeout with asyncio.timeout
# ============================================================================
async def slow_llm():
    await asyncio.sleep(10)
    return "never returned in prod"


async def demo_timeout():
    try:
        async with asyncio.timeout(1.5):  # 3.11+ context-manager form
            r = await slow_llm()
            print(r)
    except TimeoutError:
        print("timed out at 1.5s - falling back")


asyncio.run(demo_timeout())

timed out at 1.5s - falling back


In [13]:
# ============================================================================
# CONCURRENCY CONTROL: Retry with Exponential Backoff (tenacity's async API)
# ============================================================================
from tenacity import AsyncRetrying, retry_if_exception_type, stop_after_attempt, wait_exponential


class RateLimit(Exception):
    pass


_attempts = 0


async def flaky_llm(prompt: str):
    global _attempts
    _attempts += 1
    if _attempts < 3:
        raise RateLimit(f"429 (attempt {_attempts})")
    return f"ok on attempt {_attempts}"


async def demo_retry():
    async for attempt in AsyncRetrying(
        retry=retry_if_exception_type(RateLimit),
        wait=wait_exponential(multiplier=0.2, max=2),
        stop=stop_after_attempt(5),
        reraise=True,
    ):
        with attempt:
            print(await flaky_llm("hi"))


asyncio.run(demo_retry())

ok on attempt 3


In [14]:
# ============================================================================
# CONCURRENCY CONTROL: Cancellation Semantics (e.g. user closes the tab mid-run)
# ============================================================================
async def agent_step():
    try:
        await asyncio.sleep(5)
        return "done"
    except asyncio.CancelledError:
        print("  step: cancelled - flushing partial trace, closing HTTP client")
        raise  # ALWAYS re-raise CancelledError


async def demo_cancel():
    task = asyncio.create_task(agent_step())
    await asyncio.sleep(0.5)
    task.cancel()
    with contextlib.suppress(asyncio.CancelledError):
        await task


asyncio.run(demo_cancel())

  step: cancelled - flushing partial trace, closing HTTP client


---
## 👥 Part 7: Multi-Agent Supervisor with Parallel Specialist Dispatch

A supervisor dispatches independent specialist agents concurrently via `TaskGroup`, then
synthesizes their partial answers into one final response.

In [15]:
# ============================================================================
# MULTI-AGENT SUPERVISOR: Concurrent Specialist Dispatch
# ============================================================================
@dataclass
class Specialist:
    name: str
    tool: Callable

    async def run(self, q: str) -> str:
        thought = await fake_llm(f"{self.name} plan for {q}", jitter=(0.5, 1.0))
        evidence = await self.tool(q)
        return f"[{self.name}] {thought} | {evidence}"


async def supervisor(query: str):
    specialists = [
        Specialist("WebAgent", web_search),
        Specialist("RagAgent", vector_search),
        Specialist("SqlAgent", sql_query),
    ]
    t0 = time.perf_counter()
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(s.run(query)) for s in specialists]
    partials = [t.result() for t in tasks]
    synthesised = await fake_llm("synthesise: " + " \n".join(partials), jitter=(0.6, 1.0))

    print(f"supervisor elapsed: {time.perf_counter() - t0:.2f}s")
    for p in partials:
        print(" ", p)
    print("final:", synthesised)


asyncio.run(supervisor("how does Databricks Vector Search async work"))

supervisor elapsed: 2.41s
  [WebAgent] LLM_ANSWER(prompt='WebAgent plan for how does Datab') | web[how does Databricks Vector Search async work]
  [RagAgent] LLM_ANSWER(prompt='RagAgent plan for how does Datab') | vec[how does Databricks Vector Search async work]
  [SqlAgent] LLM_ANSWER(prompt='SqlAgent plan for how does Datab') | sql[how does Databricks Vector Search async work]
final: LLM_ANSWER(prompt='synthesise: [WebAgent] LLM_ANSWE')


---
## 📝 Part 8: Background Fire-and-Forget with `create_task`

Use case: log the trace to MLflow / write to a memory store - the user shouldn't have to
wait for it before getting their answer.

In [16]:
# ============================================================================
# BACKGROUND TASKS: Fire-and-Forget Logging
# ============================================================================
BACKGROUND: set[asyncio.Task] = set()  # hold refs so tasks aren't garbage-collected


async def write_trace(trace: dict):
    await asyncio.sleep(0.8)
    log.info("trace written: %s", trace)


def fire_and_forget(coro):
    t = asyncio.create_task(coro)
    BACKGROUND.add(t)
    t.add_done_callback(BACKGROUND.discard)
    return t


async def agent_with_bg_logging(q: str):
    answer = await fake_llm(q, jitter=(0.3, 0.6))
    fire_and_forget(write_trace({"q": q, "answer": answer}))
    return answer  # returns immediately - the trace flushes in the background


async def demo_bg():
    r = await agent_with_bg_logging("what is uvloop")
    print("user got:", r)
    await asyncio.gather(*BACKGROUND)  # wait at shutdown so we don't lose logs


asyncio.run(demo_bg())

user got: LLM_ANSWER(prompt='what is uvloop')


---
## 🪝 Part 9: Wrapping a Blocking Library - The Escape Hatch

Sometimes you're stuck with a sync SDK (`psycopg2`, some vendor client). `asyncio.to_thread`
offloads it to a thread pool so it doesn't freeze the event loop.

In [17]:
# ============================================================================
# ESCAPE HATCH: asyncio.to_thread for Blocking Sync Calls
# ============================================================================
def blocking_sync_call(x: int) -> int:
    time.sleep(1.0)  # sync sleep - would freeze the loop if awaited directly
    return x * x


async def demo_to_thread():
    t0 = time.perf_counter()
    outs = await asyncio.gather(*(asyncio.to_thread(blocking_sync_call, i) for i in range(5)))
    print(f"5 blocking calls, threaded gather: {time.perf_counter() - t0:.2f}s -> {outs}")


asyncio.run(demo_to_thread())

5 blocking calls, threaded gather: 1.01s -> [0, 1, 4, 9, 16]


---
## 🔑 Part 10: Real-Provider Snippets (Read-Only, Gated on Env Vars)

These cells only execute their real API call if the corresponding key / workspace is set in
`.env` - otherwise they print a skip message. They show the exact async idiom you'll use in
production with each provider.

> **Note**: Set any of these in your `.env` to actually run that cell:
> - `OPENAI_API_KEY`
> - `ANTHROPIC_API_KEY`
> - `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, `DATABRICKS_ENDPOINT`

In [18]:
# ============================================================================
# REAL PROVIDER: AsyncOpenAI Streaming
# ============================================================================
import os


async def openai_async_demo():
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not set - skipping")
        return

    from openai import AsyncOpenAI

    client = AsyncOpenAI()
    stream = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "one-line joke about event loops"}],
        stream=True,
    )
    async for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end="", flush=True)
    print()


asyncio.run(openai_async_demo())

Why did the event loop break up with the callback? It needed some space to handle its own events!


In [19]:
# ============================================================================
# REAL PROVIDER: AsyncAnthropic Streaming
# ============================================================================
async def anthropic_async_demo():
    if not os.getenv("ANTHROPIC_API_KEY"):
        print("ANTHROPIC_API_KEY not set - skipping")
        return

    from anthropic import AsyncAnthropic

    client = AsyncAnthropic()
    async with client.messages.stream(
        model="claude-3-5-sonnet-latest",
        max_tokens=200,
        messages=[{"role": "user", "content": "one-line joke about asyncio"}],
    ) as stream:
        async for text in stream.text_stream:
            print(text, end="", flush=True)
    print()


asyncio.run(anthropic_async_demo())

ANTHROPIC_API_KEY not set - skipping


In [20]:
# ============================================================================
# REAL PROVIDER: Databricks Model Serving via httpx.AsyncClient
# ============================================================================
# Env vars expected: DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_ENDPOINT
async def databricks_serving_demo():
    host = os.getenv("DATABRICKS_HOST")
    tok = os.getenv("DATABRICKS_TOKEN")
    ep = os.getenv("DATABRICKS_ENDPOINT")
    if not (host and tok and ep):
        print("Databricks env vars not set - skipping")
        return

    import httpx

    url = f"{host}/serving-endpoints/{ep}/invocations"
    payload = {
        "messages": [{"role": "user", "content": "one line: what is uvloop"}],
        "max_tokens": 100,
    }
    async with httpx.AsyncClient(timeout=30) as c:  # always use async with
        r = await c.post(url, headers={"Authorization": f"Bearer {tok}"}, json=payload)
        r.raise_for_status()
        print(r.json())


asyncio.run(databricks_serving_demo())

Databricks env vars not set - skipping


In [21]:
# ============================================================================
# REAL PROVIDER: Databricks Vector Search (sync SDK wrapped with to_thread)
# ============================================================================
# The SDK is sync; use asyncio.to_thread OR wrap it in a small async facade.
async def dbx_vector_search_demo():
    if not os.getenv("DATABRICKS_HOST"):
        print("Databricks env vars not set - skipping")
        return

    try:
        from databricks.vector_search.client import VectorSearchClient
    except ImportError:
        print("pip install databricks-vectorsearch to run this cell")
        return

    index_name = os.getenv("DBX_VS_INDEX", "main.default.my_index")

    def _query():
        vsc = VectorSearchClient()
        idx = vsc.get_index(index_name=index_name)
        return idx.similarity_search(
            query_text="async agents", columns=["id", "content"], num_results=3
        )

    result = await asyncio.to_thread(_query)
    print(result)


asyncio.run(dbx_vector_search_demo())

pip install databricks-vectorsearch to run this cell


---
## 📊 Part 11: Load Harness - How Far Does Async Take You?

Simulate N concurrent users hitting the agent, cap outbound LLM concurrency with a
semaphore, and observe how p50 / p95 latency and throughput change as load increases.

In [22]:
# ============================================================================
# LOAD HARNESS: Concurrent Users, Latency Percentiles, Throughput
# ============================================================================
import statistics


async def full_agent(q: str, sem: asyncio.Semaphore):
    t0 = time.perf_counter()
    async with sem:
        await fake_llm(f"plan {q}", jitter=(0.4, 0.8))
        async with asyncio.TaskGroup() as tg:
            tg.create_task(web_search(q))
            tg.create_task(vector_search(q))
            tg.create_task(sql_query(q))
        await fake_llm("synthesise", jitter=(0.4, 0.8))
    return time.perf_counter() - t0


async def load_test(n_users: int, concurrency_cap: int):
    sem = asyncio.Semaphore(concurrency_cap)
    t0 = time.perf_counter()
    lats = await asyncio.gather(*(full_agent(f"q{i}", sem) for i in range(n_users)))
    wall = time.perf_counter() - t0

    lats.sort()
    p50 = statistics.median(lats)
    p95 = lats[int(0.95 * len(lats)) - 1]
    print(
        f"users={n_users:3d} cap={concurrency_cap:2d} | wall={wall:5.2f}s | "
        f"p50={p50:.2f}s p95={p95:.2f}s | throughput={n_users / wall:5.1f} rps"
    )


async def sweep():
    for users, cap in [(10, 5), (25, 5), (25, 15), (50, 15)]:
        await load_test(users, cap)


asyncio.run(sweep())

users= 10 cap= 5 | wall= 5.11s | p50=3.53s p95=4.91s | throughput=  2.0 rps
users= 25 cap= 5 | wall=13.15s | p50=7.25s p95=11.54s | throughput=  1.9 rps
users= 25 cap=15 | wall= 4.75s | p50=2.57s p95=4.66s | throughput=  5.3 rps
users= 50 cap=15 | wall= 9.19s | p50=4.69s p95=8.68s | throughput=  5.4 rps


---
## 📝 Summary

In this notebook, we learned:

### 1. Core `asyncio` Primitives
- **Coroutines are lazy**: calling an `async def` function returns a coroutine object that
  does nothing until awaited or scheduled
- **`gather` vs. sequential `await`**: sequential awaits serialize independent work;
  `gather` runs it concurrently

### 2. Tool Fan-Out
- **`asyncio.gather`**: the classic way to run multiple coroutines concurrently
- **`asyncio.TaskGroup`** (3.11+): structured concurrency - one failure cancels every
  sibling task automatically, with no orphaned coroutines

### 3. Streaming
- **`async for`** over an async generator is what powers token-by-token streaming
- A stream can be paused mid-way to run a tool call, then resumed with the result folded in

### 4. Concurrency Control
- **`Semaphore`**: caps how many operations run at once, preventing rate-limit errors
- **`asyncio.timeout`**: bounds how long any single await can take
- **`tenacity.AsyncRetrying`**: retries transient failures with exponential backoff
- **Cancellation**: always re-raise `asyncio.CancelledError` after cleanup - swallowing it
  breaks task cancellation for the whole call chain

### 5. Production Patterns
- **Supervisor + specialists**: dispatch independent sub-agents concurrently via `TaskGroup`,
  then synthesize their partial results
- **Fire-and-forget**: `create_task` for background work (tracing, logging) the user
  shouldn't have to wait on - keep a reference so the task isn't garbage-collected mid-run
- **`asyncio.to_thread`**: the escape hatch for wrapping a blocking sync SDK without
  freezing the event loop
- **Load testing**: wrap the full agent in a `Semaphore`-capped `gather` sweep to see how
  p50/p95 latency and throughput degrade as concurrent users increase

### Production Checklist

| Check | Where it's demonstrated |
|---|---|
| No blocking libs in the hot path | Part 9, `asyncio.to_thread` |
| LLM SDKs use async clients | Part 10 (OpenAI / Anthropic / Databricks) |
| Tool fan-out uses `gather` / `TaskGroup` | Part 4 |
| Concurrency capped via `Semaphore` | Part 6 |
| Every external call has a timeout | Part 6, `asyncio.timeout` |
| Retry with backoff on transient errors | Part 6, `tenacity` |
| Cancellation propagates + cleans up | Part 6, `demo_cancel` |
| Streaming end-to-end | Part 5 |
| Background traces don't block the response | Part 8 |
| Load-tested with concurrent users | Part 11 |

### Next Steps
- Wrap `full_agent` in FastAPI + uvicorn (uvloop) behind a `StreamingResponse`
- Point the fake tools at real Databricks Vector Search, UC Functions, and a Model Serving endpoint
- Add MLflow tracing (`mlflow.trace`) around each `async def` - spans nest naturally with the event loop

### References
- [Databricks Agent Framework docs](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/)
- [databrickslabs GitHub](https://github.com/databrickslabs)
- [PEP 3156 - asyncio](https://peps.python.org/pep-3156/)
- [Python 3.11 TaskGroup + asyncio.timeout](https://docs.python.org/3/library/asyncio-task.html#task-groups)